In [ ]:
from pathlib import Path
import sys

REPOSITORY_ROOT = next(
    p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
    if (p / "archive/src/experiment.py").is_file()
)
PROJECT_ROOT = ROOT = REPOSITORY_ROOT / "archive"
if str(REPOSITORY_ROOT) not in sys.path:
    sys.path.insert(0, str(REPOSITORY_ROOT))


# Фиксированные ансамбли и попарные классификаторы зданий

Новых обучений нейросети здесь нет. Четыре равновесных сочетания probabilities заранее фиксированы, без сетки весов. Затем три бинарные logistic regression используют замороженные 2048 признаков ResNet101: church/palace, church/commercial_area, palace/commercial_area.

**Fitting только на train.** Стандартизация отдельно по train соответствующей пары, C=1, L2, LBFGS. sklearn/scipy не требуются. Val используется только для сравнения. Маршрутизация — исходные top-2, независимо от confidence и истинного класса. Правильно распознанные здания тоже могут попасть к классификатору: считаем fixes **и** breaks на всей val.

Существующие веса, experiments.csv и исходные predictions остаются неизменными. Результаты и обученные линейные классификаторы сохраняются в отдельных inference-run директориях. Кэш признаков содержит только train трёх классов и все val; test не читается.


In [ ]:
from pathlib import Path
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from archive.src.frozen_experiments import run_fixed_ensembles, run_pairwise
from archive.src.pairwise import extract_building_features

DEVICE = 'auto'
CPU_THREADS = 4
BATCH_SIZE = 8
ENSEMBLE_RUN_ID = 'frozen_probability_ensembles_001'
PAIRWISE_RUN_ID = 'frozen_building_reranker_001'
FEATURE_DIR = 'artifacts/predictions/building_pairwise_features_001'
CHECKPOINT = 'artifacts/models/resnet101_partial_domain_shift_stronger_001/best.pt'


## 1. Четыре фиксированных probability ensembles

Reference: mean(ResNet101 none, stronger50 D4), 679/699. Добавление CosFace/mining означает равные веса трёх исходных моделей, а не 50% новому участнику. D4 считается одним участником. fixes/breaks в таблице относительно reference.

In [ ]:
ensemble_results = run_fixed_ensembles(PROJECT_ROOT, ENSEMBLE_RUN_ID)
display(ensemble_results)


## 2. Замороженные признаки

Первый запуск делает CPU/GPU inference на ~750 train зданий и 699 val. Завершённый кэш повторно используется с проверкой hashes. Никаких аугментаций, TTA или градиентов backbone.

In [ ]:
feature_path = extract_building_features(PROJECT_ROOT, CHECKPOINT, FEATURE_DIR,
    device=DEVICE, batch_size=BATCH_SIZE, cpu_threads=CPU_THREADS)
display(json.loads((feature_path/'metadata.json').read_text(encoding='utf-8')))


## 3. Train-only logistic regression и top-2 reranking

Одни и те же три классификатора применяем к одиночной 101 и прежнему лучшему ансамблю. Пороги/веса по val не подбираются. Результат сохраняется как labels, без выдуманного 20-классового распределения вероятностей.

In [ ]:
pairwise_results = run_pairwise(PROJECT_ROOT, FEATURE_DIR, PAIRWISE_RUN_ID)
display(pairwise_results)
pair_logs = PROJECT_ROOT/'artifacts/logs'/PAIRWISE_RUN_ID
display(pd.read_csv(pair_logs/'binary_diagnostics.csv'))
display(pd.read_csv(pair_logs/'routing_summary.csv'))


## 4. Сравнение результатов и всех изменений

In [ ]:
comparison = pd.concat([ensemble_results[['recipe','correct','errors','val_accuracy']],
                        pairwise_results[['recipe','correct','errors','val_accuracy']]], ignore_index=True)
fig, ax = plt.subplots(figsize=(12,5))
ax.barh(comparison.recipe, comparison.correct, color=['#3973ac']*len(ensemble_results)+['#b05f32']*len(pairwise_results))
ax.axvline(679,color='#238b45',linestyle='--',label='Reference: 679')
ax.set(xlim=(min(comparison.correct)-3, max(comparison.correct)+2), xlabel='Correct / 699')
ax.legend(); fig.tight_layout(); plt.show()
for base in ['resnet101_single','reference_101_50d4']:
    print(base)
    display(pd.read_csv(pair_logs/f'{base}_changes.csv'))
    display(pd.read_csv(pair_logs/f'{base}_classes.csv').query('errors > 0'))


## 5. Изображения, на которых reranker поменял ответ

Показываем все изменения, включая регрессии, без выбора только исправленных объектов.

In [ ]:
from PIL import Image
for base in ['resnet101_single','reference_101_50d4']:
    changes = pd.read_csv(pair_logs/f'{base}_changes.csv', dtype={'id':str})
    if changes.empty:
        print(base, 'no changed labels'); continue
    for start in range(0,len(changes),9):
        page = changes.iloc[start:start+9]
        fig,axes = plt.subplots((len(page)+2)//3,3,figsize=(15,4*((len(page)+2)//3)),squeeze=False)
        for ax,row in zip(axes.flat,page.itertuples()):
            directory=(PROJECT_ROOT/'data/raw/val').resolve()
            image_path=(directory/row.id).resolve()
            if not image_path.is_relative_to(directory): raise ValueError('ID escapes val')
            with Image.open(image_path) as image: ax.imshow(image.convert('RGB'))
            ax.set_title(f'{row.id} | {row.change} | true: {row.true_class}\n{row.reference_class} -> {row.predicted_class}',fontsize=10)
            ax.axis('off')
        for ax in list(axes.flat)[len(page):]: ax.axis('off')
        fig.suptitle(base); fig.tight_layout(); plt.show()


## После выбора pipeline

Сначала сравнить общий баланс fixes/breaks, особенно на church/palace/commercial_area. Не выбирать отдельные ошибки вручную и не подбирать порог на каждом снимке. Если нет прироста, оставить reference.

По подтверждённым пользователем правилам финальный train+val refit разрешён. Он вынесен в **09_final_refit.ipynb**, отключён по умолчанию и требует Kaggle GPU. После него бывшая val не служит held-out метрикой или источником выбора checkpoint. Подготовленные train+val presets — финальное обучение выбранного partial+stronger, не ещё один full-контроль.
